In [ ]:
!pip install openpyxl

V3_MEJORADO

In [1]:
"""
================================================================================
CLASIFICADOR DE SENTIMIENTO POLÍTICO PERUANO — VERSIÓN 3 (BALANCEADA)
================================================================================
CORRECCIÓN 10 — Balance estructural de reglas:

El conjunto de reglas que suman a pos_score (anti-Castillo/pro-Keiko,
mapeado a +1) tenía 38 patrones activos, frente a solo 22 patrones que
suman a neg_score (anti-Keiko/pro-Castillo, mapeado a -1). Esta asimetría
de 16 patrones no respondía a una diferencia real en el discurso, sino a
que varios patrones menores y de baja especificidad (burlas sobre la forma
de hablar, muletillas, correcciones de hechos, promesas irreales, elogios
étnicos, referencias a Evo Morales, bonos, jerga positiva suelta, etc.)
solo tenían versión "pro-Keiko" y ningún patrón equivalente del lado
"pro-Castillo". En tweets ambiguos, esto daba al sistema estructuralmente
más caminos para acumular pos_score que neg_score, sesgando el promedio
diario hacia el polo +1 con mayor frecuencia de la que el discurso
justificaría, lo que dificultaba que el índice cruzara el umbral necesario
para predecir correctamente los días de baja del tipo de cambio (baja
especificidad, 0.21, en la validación de la Tabla 11).

Se retiran los 12 patrones exclusivos y de menor especificidad del lado
pos_score sin contraparte: BURLA_FORMA_HABLAR, MULETILLA_REPETICION,
CORRECCION_FALSA, RECOGE_TAPER, SIN_EQUIPO_TECNICO, ELOGIO_KEIKO_ETNICO,
PROMESAS_IRREALES, ECONOMIA_ESPECIFICA, SLANG_POSITIVO,
ELOGIO_FUJIMORI_PADRE, BONOS_KEIKO, EVO_ADVERTENCIA. El conteo de patrones
activos queda en 26 (pos_score) vs 22 (neg_score), balance cercano a 1:1
frente al 38 vs 22 original, sin tocar los patrones de alta especificidad
que sustentan el aporte metodológico central del sistema.

Todo lo demás (Correcciones 1-9) se mantiene igual.
================================================================================
"""

import pandas as pd
import numpy as np
import re
from typing import Dict, Optional
import warnings
warnings.filterwarnings('ignore')


class PeruvianSentimentClassifier:

    MARGEN_MINIMO = 0.15

    def __init__(self, use_model: bool = True):
        self.model_loaded = False
        self.cache = {}
        self.metrics = {
            'total': 0, 'model_used': 0, 'rules_only': 0,
            'hybrid': 0, 'filtered': 0, 'neutral_real': 0,
            'political_override': 0, 'neutral_margen_bajo': 0
        }
        self._compile_patterns()
        if use_model:
            self._load_model()

    def _load_model(self):
        try:
            from transformers import pipeline
            print("Cargando modelo ML...")
            self.sentiment_pipeline = pipeline(
                "sentiment-analysis",
                model="cardiffnlp/xlm-twitter-politics-sentiment",
                max_length=512,
                truncation=True,
                top_k=None
            )
            self.model_loaded = True
            print("✓ Modelo cargado")
        except Exception:
            print("⚠ Modelo no disponible. Usando solo reglas.")
            self.model_loaded = False

    NEGACION = re.compile(
        r'\b(no\s+es|no\s+creo|no\s+es\s+cierto|mentira\s+que|falso\s+que|'
        r'no\s+soy|nunca\s+dij[eo]|no\s+dij[eo]|no\s+se[ñn]al[oó]|'
        r'no\s+ser[íi]a|jam[aá]s\s+dij[eo]|no\s+creas|no\s+creamos|'
        r'ni\s+de\s+broma\s+es|para\s+nada\s+es)\b', re.I
    )

    def _hay_negacion_cerca(self, texto, pos_inicio, ventana=40):
        inicio_ventana = max(0, pos_inicio - ventana)
        return bool(self.NEGACION.search(texto[inicio_ventana:pos_inicio]))

    URL_PATTERN = re.compile(r'https?://\S+')
    MENTION_PATTERN = re.compile(r'@\w+')

    def _contenido_insuficiente(self, texto, min_palabras=3):
        sin_menciones = self.MENTION_PATTERN.sub('', texto)
        sin_urls = self.URL_PATTERN.sub('', sin_menciones)
        palabras_reales = [w for w in sin_urls.split() if len(w) > 1]
        return len(palabras_reales) < min_palabras

    def _compile_candidate_lists(self):
        self.NOMBRES_MERCADO = re.compile(
            r'\b(keiko|fujimori|fujimorismo|fuerza\s+popular|la\s+china|'
            r'señora\s*k|l[oó]pez\s+aliaga|rla|hernando\s+de\s+soto|de\s+soto|'
            r'sheput)\b', re.I
        )
        self.NOMBRES_CAMBIO = re.compile(
            r'\b(castillo|pedro\s+castillo|profesor|profe|perú\s+libre|'
            r'cerrón|vladimir\s+cerrón|ver[oó]nika|mendoza|yonhy\s+lescano|'
            r'lescano)\b', re.I
        )

    def _resolver_direccion(self, texto, pos_match, ventana=80):
        inicio = max(0, pos_match - ventana)
        fin = min(len(texto), pos_match + ventana)
        contexto = texto[inicio:fin]
        match_mercado = self.NOMBRES_MERCADO.search(contexto)
        match_cambio = self.NOMBRES_CAMBIO.search(contexto)
        if match_mercado and match_cambio:
            pos_patron_en_contexto = pos_match - inicio
            dist_mercado = abs(match_mercado.start() - pos_patron_en_contexto)
            dist_cambio = abs(match_cambio.start() - pos_patron_en_contexto)
            return 'MERCADO' if dist_mercado < dist_cambio else 'CAMBIO'
        elif match_mercado:
            return 'MERCADO'
        elif match_cambio:
            return 'CAMBIO'
        return None

    def _detectar_bando_mencionado(self, texto):
        hay_mercado = bool(self.NOMBRES_MERCADO.search(texto))
        hay_cambio = bool(self.NOMBRES_CAMBIO.search(texto))
        if hay_mercado and hay_cambio:
            return 'AMBOS'
        elif hay_mercado:
            return 'MERCADO'
        elif hay_cambio:
            return 'CAMBIO'
        return None

    def _reorientar_polaridad_modelo(self, texto, sentiment_modelo):
        if sentiment_modelo not in ('POSITIVE', 'NEGATIVE'):
            return sentiment_modelo
        bando = self._detectar_bando_mencionado(texto)
        if bando == 'CAMBIO':
            return 'NEGATIVE' if sentiment_modelo == 'POSITIVE' else 'POSITIVE'
        return sentiment_modelo

    def _compile_patterns(self):
        self._compile_candidate_lists()

        self.CRITICA_KEIKO = re.compile(
            r'\b(keiko|fujimori|fujimorismo|fuerza\s+popular|señorak|señora\s+k|la\s+k).{0,80}'
            r'(corrupta|ladrona?|criminal|dictadora?|mafia|cárcel|presa?|'
            r'prisión|delincuente|narco|roba|robó|cleptócrata|naranja|'
            r'fujirata|cleptocracia|condenada|procesada|chori|fujitroll|fujibot|'
            r'autoritaria|poco\s+aprecio)\b|'
            r'\b(no\s+a\s+keiko|keiko\s+no|keiko\s+nunca|fujimorismo\s+nunca|'
            r'fujimori\s+nunca\s+más|keikonomás|keikono|kno)\b|'
            r'\#(fujimorinuncamas|keikonova|keikono|noakeiko)\b|'
            r'\bk\s+(no|nunca|jamás)\b|'
            r'\b(chino|china).{0,30}(corrupta|ladrona|criminal)\b|'
            r'\broba\s+pero\s+hace\s+obra\b|'
            r'\bplata\s+como\s+cancha\b|'
            r'\bnarco(indultos?|política)\b|'
            r'\bla\s+china\b|\bseñorak\b', re.I
        )
        self.TEMAS_FUJIMORI = re.compile(
            r'\b(esterilizacion(es)?(\s+forzadas?)?|autogolpe|montesinos|'
            r'dictadura|años\s+90|década\s+del\s+90|90s|'
            r'lavado\s+de\s+activos|narcoindulto|cuellos?\s+blancos?|'
            r'vladimiro|corrupción\s+fujimori|barrios\s+altos|la\s+cantuta|'
            r'grupo\s+colina|diarios\s+chicha|naranja|uniformes?\s+naranjas?)\b', re.I
        )
        self.APOYO_CASTILLO = re.compile(
            r'\b(voto|votaré|apoyo|respaldo|con|vamos|fuerza|arriba)\s+(por\s+)?(pedro|castillo|profesor|profe)\b|'
            r'\b(pedro|castillo)\s+(presidente|al\s+poder|sí|ya|2021)\b|'
            r'\bperú\s+libre\b|'
            r'\blápiz\s+(sí|si|ganador|presidente)\b|'
            r'\b(cambio|esperanza|pueblo|dignidad).{0,30}(castillo|pedro)\b|'
            r'\bpedro\s+castillo\s+(noqueó|ganó|fue\s+mejor|presidente\s+y\s+punto\s+final)\b|'
            r'\byo\s+escojo.{0,20}castillo\b|'
            r'\bmás\s+limpio\b', re.I
        )
        self.VICTORIA_CASTILLO = re.compile(r'\bsalió\s+trasquilada\b|\bfue\s+más\b', re.I)
        self.MOVILIZACION_PROTESTA = re.compile(r'\btomar\s+las\s+calles\b|\bno\s+somos\s+terrucos\b', re.I)
        self.ACUSACION_TRAMPA_KEIKO = re.compile(
            r'\bayuda\s+auricular\b|\baudífono\b|\bkeiko.{0,30}trampa\b', re.I
        )
        self.IZQUIERDA_REFS = re.compile(
            r'\b(cerrón|vladimir\s+cerrón|verónika|mendoza|anahí|durand|'
            r'asamblea\s+constituyente|nueva\s+constitución|'
            r'patria\s+roja|movadef|san\s+indira|indira\s+huilca)\b', re.I
        )
        self.INSULTOS_CASTILLO = re.compile(
            r'\b(tirapiedras?|tira\s+piedras?|castillo.{0,20}piedra|'
            r'rondero|campesino\s+ignorante|analfabeto|sin\s+estudios|'
            r'profesor.{0,30}(ignorante|bruto|improvisado)|'
            r'maestro.{0,30}(ignorante|bruto|improvisado)|'
            r'profe.{0,20}(bruto|ignorante)|lapicito|lapicero|pedrito|'
            r'chotano|provinc(iano|iana)|maestro\s+ignorante|'
            r'conchudazo|conchudo|porky)\b|'
            r'\b(cholo|serrano).{0,30}(ignorante|bruto|improvisado)\b', re.I
        )
        self.APOYO_KEIKO = re.compile(
            r'\b(voto|votaré|apoyo|respaldo|con|vamos|fuerza)\s+(por\s+)?keiko\b|'
            r'\bkeiko\s+(presidenta|presidente|2021|sí|ya)\b|'
            r'\bfuerza\s+(keiko|popular)\b|'
            r'\bmal\s+menor\b|'
            r'\bkeiko.{0,30}(ganó|destruyó|barrió|apabulló)\b|'
            r'\basí\s+es.{0,10}keiko\b', re.I
        )
        self.CRITICA_CASTILLO = re.compile(
            r'\b(castillo|pedro|cerrón|perú\s+libre|lápiz).{0,80}'
            r'(terrorista|comunista|ignorante|improvisado|bruto|'
            r'chavista|venezuela|cuba|terruco|sendero|narco|zurdos?|caviares?|'
            r'extremista|radical|cerronista|comunacho|cobardemente|vista\s+gorda)\b|'
            r'\b(no\s+a\s+castillo|castillo\s+no|pedro\s+no|castillo\s+nunca)\b|'
            r'\bperú\s+libre.{0,50}(terrorismo|comunismo|venezuela|sendero)\b|'
            r'\bcaviar(es)?\b|'
            r'\bcómo\s+es\s+que.{0,20}devuelve.{0,20}robado\b|'
            r'\bdevuelve.{0,20}plata\b|'
            r'\bcarece.{0,20}condiciones\s+mínimas\b|'
            r'\bdirás\s+que.{0,20}(cerrón|vladimir).{0,20}santo\b', re.I
        )
        self.ANTI_COMUNISMO = re.compile(
            r'\b(no\s+al\s+comunismo|peligro\s+comunista|fuera\s+socialismo|'
            r'venezuela\s+no|cuba\s+no|no\s+seré\s+venezuela|no\s+seremos\s+venezuela|'
            r'comunismo\s+nunca|rechazo\s+comunismo|chavismo|madurismo|'
            r'zurdo|zurdos|más\s+socialismo|estatista|'
            r'control\s+de\s+precios|expropiación|expropriac|'
            r'izquierda\s+miserable|izquierdamiserable|'
            r'ya\s+fueeee|ya\s+fue|comunismo\s+nunca\s+será\s+gobierno|'
            r'a\s+llorar\s+al\s+río)\b', re.I
        )
        self.TERRORISMO = re.compile(
            r'\b(sendero(\s+luminoso)?|terrorismo|terrorista|movadef|'
            r'terruco|apología\s+al\s+terrorismo|vínculos?\s+terroristas?|'
            r'abimael|guzmán|senderista)\b', re.I
        )
        self.ESTABILIDAD_ECONOMICA = re.compile(
            r'\b(estabilidad\s+económica|inversión\s+privada|'
            r'libre\s+mercado|economía\s+de\s+mercado|'
            r'mal\s+menor|menos\s+peor|votaré\s+en\s+contra|'
            r'para\s+que\s+no\s+gane|impedir\s+que\s+gane|'
            r'confianza\s+económica|mercados)\b', re.I
        )
        self.VENEZUELA_CUBA = re.compile(
            r'\b(venezuela|cuba).{0,60}(crisis|colapso|hambre|dictadura|'
            r'represión|comunismo|maduro|chávez|castro|miseria|pobreza|sueldo mínimo)\b|'
            r'\bno\s+quiero\s+(venezuela|cuba)\b|'
            r'\b(chavismo|chavista|castrismo|castrista)\b|'
            r'\bte\s+invito\s+a\s+ir\s+venezuela\b', re.I
        )
        self.MIEDO_ECONOMICO = re.compile(
            r'\b(crisis\s+económica|colapso\s+económico|caos\s+económico|'
            r'fuga\s+de\s+capitales|dólar\s+(sube|dispara|explota|se\s+va|a\s+\d+)|'
            r'inflación|hiperinflación|recesión|devaluación|'
            r'empresas\s+se\s+van|economía\s+se\s+hunde|'
            r'bolsa\s+cae|riesgo\s+país|default|'
            r'cachetada\s+a\s+la\s+economía)\b', re.I
        )
        self.ERROR_FACTUAL = re.compile(
            r'\bal\s+enterarse\b|\bresultó\s+que\b|\bse\s+enteró\b|'
            r'\bmachu\s+picchu.{0,30}(no\s+tiene|tiene).{0,20}aeropuerto\b|'
            r'\ben\s+verdad.{0,20}dijo\b|\bde\s+verdad.{0,20}dijo\b|'
            r'\ben\s+serio.{0,20}dijo\b', re.I
        )
        self.SARCASMO_IRONIA = re.compile(
            r'\bahí\s+lo\s+tienen\b|\bahí\s+está\b|'
            r'\ben\s+todo\s+su\s+esplendor\b|'
            r'\bse\s+fumó\b|\bestá\s+fumado\b|\bqué\s+fumó\b|'
            r'\brelaaaax\b|'
            r'\bya\s+será\s+el\s+ganador\b', re.I
        )
        self.PREGUNTA_RETORICA_CRITICA = re.compile(
            r'\bpresentará\s+propuestas?\s+o.{0,30}atacar\b|'
            r'\bentonces\s+(qué|que).{0,30}(regalarán|gratis|cobrarán)\b|'
            r'\bo\s+sea\s+que.{0,30}(regalarán|gratis)\b|'
            r'\bse\s+dedicará\s+a\s+atacar\b|'
            r'\bqué\s+debate\s+viste\b|\bque\s+debate\s+vio\b|'
            r'\bviste\s+otro\s+debate\b|\ben\s+qué\s+debate\b', re.I
        )
        self.EVASION_RESPUESTA = re.compile(
            r'\bevade\b|\bevadió\b|\bno\s+responde\b|'
            r'\bcambia\s+de\s+tema\b|\bhabla\s+de\s+otra\s+cosa\b|'
            r'\blejos\s+de\b|\bno\s+está\s+enfocado\b|'
            r'\bno\s+te\s+corras\b|\bse\s+corre\s+de\b', re.I
        )
        self.VOTO_CASTIGO = re.compile(
            r'\b(voto\s+castigo|votaré\s+en\s+contra|para\s+que\s+no\s+gane|'
            r'impedir\s+que|menos\s+peor|mal\s+menor)\b', re.I
        )
        self.ORDEN_CAOS = re.compile(
            r'\borden.{0,20}caos\b|\bkeiko.{0,30}orden\b|\bcastillo.{0,30}caos\b', re.I
        )
        self.AUTORITARIO_DESPIADADO = re.compile(
            r'\bhombre\s+duro\b|\bdespiadado\b|'
            r'\bcontrolar\s+a\s+la\s+población\b|'
            r'\bmanejar.{0,20}personas\b', re.I
        )
        self.PAIS_DESARROLLADO_SARCASMO = re.compile(
            r'\bperú\s+es\s+(dubai|new\s+york|suiza)\b|'
            r'\bseguro\s+estamos\s+en\s+(dubai|suiza)\b|'
            r'\bsomos\s+(dubai|suiza)\b|'
            r'\bestamos\s+en\s+(dubai|suiza)\b', re.I
        )
        self.ERROR_MATEMATICO = re.compile(
            r'\bimplementar\s+el\s+pbi\b|'
            r'\belevar\s+el\s+pbi\b|'
            r'\bla\s+matemática\s+no\s+cuadra\b|'
            r'\b10x10\b|'
            r'\bcurso.{0,20}economía\s+básica\b|'
            r'\b%.{0,10}pbi.{0,20}educación.{0,20}%.{0,10}salud\b', re.I
        )
        self.BUSCANDO_PUESTO = re.compile(
            r'\bbuscando\s+(chamba|ministerio|puesto)\b|'
            r'\bquieres\s+ser\s+ministro\b|'
            r'\bya\s+quieres\s+ministerio\b|'
            r'\bsobando\b|\bmendigar\b', re.I
        )
        self.OPORTUNISTA = re.compile(
            r'\bcomodín\b|\barrimado\b|\bcamaleón\b|'
            r'\bfelpudo\b|\bsobón\b|\balcahuete\b|'
            r'\blamebotas\b|\blamecu\b|\bhuele\s+pedos\b', re.I
        )
        self.HASHTAG_FUJIMORISMO_NUNCA = re.compile(
            r'\bfujimoris?mo\s?nunca\s?m[aá]s\b|'
            r'\#fujimoris?monuncamas\b|'
            r'\bfujimori\s?nunca\s?m[aá]s\b', re.I
        )
        self.APRENDE_PERDER = re.compile(
            r'\baprende\s+a\s+perder\b|\bmal\s+perdedor\b|\bno\s+sabes?\s+perder\b', re.I
        )
        self.MOCHILA_PASADO = re.compile(
            r'\b(mochila|pasado|historial).{0,30}(oscur[oa]|pesad[oa]|negr[oa]|sucio|corrupto)\b|'
            r'\b(oscur[oa]|pesad[oa]).{0,30}mochila\b', re.I
        )
        self.JAPONESA_DESPECTIVO = re.compile(r'\bla\s+japonesa\b|\bjaponesa\s+(no|nunca|jamás)\b', re.I)
        self.SACAR_VUELTA_LEY = re.compile(
            r'\bsacar(le)?\s+(la\s+)?vuelta\s+(a\s+)?la\s+ley\b|'
            r'\bevadir\s+(la\s+)?ley\b|'
            r'\bburlarse\s+de\s+la\s+ley\b', re.I
        )
        self.ASOCIACION_PERDER = re.compile(
            r'\bapoya.{0,30}(keiko|fujimori).{0,30}(perder|va\s+perder|pierde|perdió)\b|'
            r'\b(keiko|fujimori).{0,30}va\s+perder\b', re.I
        )
        self.PREGUNTA_RETORICA_PROPUESTAS = re.compile(
            r'\b(cu[aá]l|qu[eé]).{0,30}(mejor|buena).{0,30}propuesta.{0,30}(castillo|pedro)\b|'
            r'\b(castillo|pedro).{0,30}(cu[aá]l|qu[eé]).{0,30}propuesta\b|'
            r'\bqu[eé]\s+propone\s+(castillo|pedro)\b', re.I
        )
        self.ERROR_PBI_PORCENTAJE = re.compile(
            r'\b%\s+(del\s+)?pbi.{0,30}educaci[oó]n.{0,30}%\s+(del\s+)?pbi.{0,30}salud\b|'
            r'\b(implementar|elevar|subir)\s+(el\s+)?pbi\b|'
            r'\bno\s+tiene\s+las\s+ideas\s+(m[aá]s\s+)?b[aá]sicas\b|'
            r'\bmatem[aá]tica\s+no\s+cuadra\b', re.I
        )
        self.VICTORIA_KEIKO_DEBATE = re.compile(
            r'\bkeiko.{0,30}(barri[oó]|destru[iy][oó]|gan[oó]|apabull[oó]).{0,30}(castillo|pedro)\b|'
            r'\bkeiko.{0,30}fue\s+(mejor|superior)\b', re.I
        )
        self.OPORTUNISTA_CASTILLO = re.compile(
            r'\b(castillo|pedro).{0,30}(aprovechar|aprovech[oó]|oportunista)\b|'
            r'\biba\s+aprovechar.{0,30}castillo\b', re.I
        )
        self.LUCHADOR_SOCIAL = re.compile(
            r'\b(castillo|pedro).{0,30}luchador\s+social\b|'
            r'\bluchador\s+social.{0,30}(castillo|pedro)\b|'
            r'\brepresenta\s+al\s+pueblo\b', re.I
        )
        self.FELPUDO_INSULTOS = re.compile(r'\bfelpud[oi]\b|\bfelpudin[oi]\b|\blamebotas\b|\blamecu\b', re.I)

        self.EMOCION_NEGATIVA = re.compile(
            r'\b(miedo|terror|pánico|asusta|temo|dios\s+nos\s+libre|'
            r'nos\s+vamos\s+a\s+(hundir|joder|caer)|estamos\s+(perdidos|jodidos|fritos)|'
            r'odio|asco|cólera|rabia|me\s+da\s+asco|vergüenza|vergonzoso|'
            r'qué\s+vergüenza|da\s+vergüenza|indignante|indigna|repugnante)\b', re.I
        )
        self.INSULTOS_GENERALES = re.compile(
            r'\b(tonto|tonta|bruto|bruta|burro|burra|ignorante|estúpido|estúpida|'
            r'imbécil|idiota|mentiroso|mentirosa|charlatán|charlatan|demagogo|demagoga|'
            r'corrupto|corrupta|ladrón|ladrona|delincuente|criminal|'
            r'racista|xenófobo|fascista)\b', re.I
        )
        self.SLANG_NEGATIVO = re.compile(
            r'\b(webada|huevada|cojudez|pendejada|mierda|chucha|carajo|'
            r'puta|csm|ptm|ctm|hdp|vale\s+mierda|qué\s+(chucha|mierda)|'
            r'la\s+cagó|cagaste|nos\s+cagaron|gil|cojudo|cojuda|pendejo|pendeja|'
            r'huevón|huevona|misio|vendido|vendida|rata|sapo|sapa|'
            r'tmr|asu\s+mare|reconcha|basura|porquería)\b', re.I
        )
        self.SIN_PROPUESTAS = re.compile(
            r'\b(sin\s+propuestas?|no\s+(tiene\s+)?propone|nada\s+de\s+nada|'
            r'puro\s+(floro|palabreo|diagnóstico|humo)|pura\s+palabrería|'
            r'(mucho|tanto|puro|solo|sólo)\s+floro|solo\s+florea|'
            r'cero\s+(propuestas?|planes?)|ninguna?\s+propuestas?|'
            r'se\s+dedica\s+a\s+atacar|puro\s+ataque|vacía|vacío)\b', re.I
        )
        self.NO_PREPARADO = re.compile(
            r'\b(improvisado|improvisada|improvisación|'
            r'sin\s+preparación|no\s+está\s+preparado|no\s+sabe\s+nada|'
            r'no\s+tiene\s+idea|falta\s+de\s+preparación|'
            r'no\s+sabe\s+(de|del)|desconoce|ignorante\s+de)\b', re.I
        )
        self.CRITICA_ECONOMICA = re.compile(
            r'\b(inviable|imposible|no\s+funciona|no\s+sirve|no\s+puede|'
            r'fracaso|fracasará|utopía|utópico|insostenible|'
            r'irrealizable|fantasía|fantasioso|populismo|populista)\b', re.I
        )
        self.PERMISO_PODER_JUDICIAL = re.compile(
            r'\bpedir\s+permiso.{0,20}(poder\s+judicial|juez|fiscal)\b|'
            r'\bpermiso\s+al\s+(poder\s+judicial|juez)\b', re.I
        )
        self.POPULISMO_KEIKO = re.compile(
            r'\bde\s+dónde.{0,30}(dinero|plata|sacará)\b|'
            r'\btanto.{0,20}bono\b|'
            r'\bpuro\s+populismo\b|'
            r'\bkeiko.{0,40}populismo\b', re.I
        )
        self.PUEBLO_CASTILLO = re.compile(
            r'\bpueblo.{0,30}(castillo|pedro|dignidad|libre)\b|'
            r'\b(castillo|pedro).{0,30}pueblo\b', re.I
        )
        self.POBRES_CRITICA = re.compile(r'\bpobres.{0,40}(abandonados|olvidados|keiko|fujimori)\b', re.I)
        self.CRITICA_INSTITUCIONAL_TERM = re.compile(
            r'\bmafia\b|'
            r'\bautoritaria\s+como\s+su\s+padre\b|'
            r'\bjne.{0,30}celeridad\b|'
            r'\bel\s+único\s+fin.{0,30}atacar\b|'
            r'\bnadie\s+se\s+la\s+cree\b', re.I
        )
        self.BURLA_RISA_TERM = re.compile(r'\b(jaja|jeje|jiji|ajaja|jajaja|jajaj|jjaja|haha)\b', re.I)

        self.APOYO_MENDOZA = re.compile(
            r'\b(voto|votaré|apoyo|respaldo|con|vamos|fuerza)\s+(por\s+)?'
            r'(ver[oó]nika|mendoza)\b|'
            r'\b(ver[oó]nika|mendoza)\s+(presidenta|2021|sí|ya)\b|'
            r'\bjuntos\s+por\s+el\s+per[uú]\b', re.I
        )
        self.CRITICA_MENDOZA = re.compile(
            r'\b(ver[oó]nika|mendoza).{0,60}(comunista|chavista|venezuela|radical|extremista|zurda)\b', re.I
        )
        self.APOYO_RLA_DESOTO = re.compile(
            r'\b(voto|votaré|apoyo|respaldo|con|vamos|fuerza)\s+(por\s+)?'
            r'(rla|l[oó]pez\s+aliaga|hernando\s+de\s+soto|de\s+soto)\b|'
            r'\b(rla|l[oó]pez\s+aliaga|de\s+soto)\s+(presidente|2021|sí|ya)\b|'
            r'\brenovaci[oó]n\s+popular\b|'
            r'\bmi\s+voto\s+es\s+por\s+rla\b', re.I
        )
        self.CRITICA_LESCANO = re.compile(
            r'\byonhy\s+lescano.{0,60}(cabestrillo|incoherente|inconsistente)\b', re.I
        )
        self.ANALISIS_TECNICO = re.compile(
            r'\b(según\s+(la\s+)?encuesta|datos\s+de|estadísticas?|ipsos|iep|'
            r'el\s+debate\s+técnico|propuestas?\s+de|plan\s+de\s+gobierno|'
            r'programa\s+electoral|análisis\s+de|comparación\s+de|'
            r'ambos\s+candidatos|los\s+dos\s+candidatos|primera\s+vuelta|segunda\s+vuelta|'
            r'onpe|reniec|jne|boca\s+de\s+urna|conteo\s+rápido|'
            r'flash\s+electoral|voto\s+informado|fact\s+checking)\b', re.I
        )
        self.PREGUNTA_GENUINA = re.compile(
            r'\b(cuál\s+es\s+(la\s+)?propuesta|qué\s+propone|'
            r'me\s+pueden\s+explicar|alguien\s+sabe|'
            r'dónde\s+puedo\s+ver|información\s+sobre|'
            r'cuáles\s+son\s+las\s+diferencias|voto\s+(nulo|blanco))\b', re.I
        )

        self.STRONG_POLITICAL_PATTERNS = [
            self.CRITICA_KEIKO, self.CRITICA_CASTILLO,
            self.APOYO_KEIKO, self.APOYO_CASTILLO,
            self.TERRORISMO, self.ANTI_COMUNISMO,
            self.VENEZUELA_CUBA, self.INSULTOS_CASTILLO,
            self.TEMAS_FUJIMORI, self.APOYO_RLA_DESOTO,
            self.APOYO_MENDOZA
        ]

    def _normalize_text(self, text):
        if not isinstance(text, str):
            return ""
        return text.strip().lower()

    def _get_political_content_strength(self, text):
        political_matches = sum(1 for p in self.STRONG_POLITICAL_PATTERNS if p.search(text))
        max_possible = len(self.STRONG_POLITICAL_PATTERNS)
        return political_matches / max_possible if max_possible > 0 else 0

    def _sumar_con_direccion(self, texto, patron, peso, nombre_razon, neg_score, pos_score, reasons):
        match = patron.search(texto)
        if not match:
            return neg_score, pos_score
        if self._hay_negacion_cerca(texto, match.start()):
            return neg_score, pos_score
        direccion = self._resolver_direccion(texto, match.start())
        if direccion == 'CAMBIO':
            pos_score += peso
            reasons.append(f'{nombre_razon}_vs_CASTILLO')
        elif direccion == 'MERCADO':
            neg_score += peso
            reasons.append(f'{nombre_razon}_vs_KEIKO')
        return neg_score, pos_score

    def analyze_rules(self, text):
        normalized = self._normalize_text(text)
        words = normalized.split()

        if len(words) < 2 or len(normalized) < 10:
            return {'sentiment': 'NEUTRAL', 'confidence': 0.0, 'score_neg': 0,
                    'score_pos': 0, 'score_neutral': 0, 'reason': 'TEXTO_CORTO',
                    'political_strength': 0.0}

        neg_score = 0.0
        pos_score = 0.0
        neutral_score = 0.0
        reasons = []

        # --- neg_score: 22 patrones ---
        if self.CRITICA_KEIKO.search(normalized): neg_score += 6.5; reasons.append('CRITICA_KEIKO')
        if self.TEMAS_FUJIMORI.search(normalized): neg_score += 5.5; reasons.append('TEMAS_FUJIMORI')
        if self.APOYO_CASTILLO.search(normalized): neg_score += 5.5; reasons.append('APOYO_CASTILLO')
        if self.VICTORIA_CASTILLO.search(normalized): neg_score += 5.0; reasons.append('VICTORIA_CASTILLO')
        if self.PERMISO_PODER_JUDICIAL.search(normalized): neg_score += 4.5; reasons.append('PERMISO_PJ')
        if self.MOVILIZACION_PROTESTA.search(normalized): neg_score += 4.0; reasons.append('MOVILIZACION')
        if self.ACUSACION_TRAMPA_KEIKO.search(normalized): neg_score += 3.5; reasons.append('TRAMPA_KEIKO')
        if self.POPULISMO_KEIKO.search(normalized): neg_score += 3.5; reasons.append('POPULISMO_KEIKO')
        if self.PUEBLO_CASTILLO.search(normalized): neg_score += 3.5; reasons.append('PUEBLO_CASTILLO')
        if self.POBRES_CRITICA.search(normalized): neg_score += 3.0; reasons.append('POBRES_CRITICA')
        if self.NO_PREPARADO.search(normalized): neg_score += 3.5; reasons.append('NO_PREPARADO')
        if self.SIN_PROPUESTAS.search(normalized): neg_score += 3.5; reasons.append('SIN_PROPUESTAS')
        if self.CRITICA_ECONOMICA.search(normalized): neg_score += 3.0; reasons.append('CRITICA_ECON')
        if self.IZQUIERDA_REFS.search(normalized): neg_score += 3.0; reasons.append('IZQUIERDA')
        if self.HASHTAG_FUJIMORISMO_NUNCA.search(normalized): neg_score += 8.0; reasons.append('FUJIMORI_NUNCA_MAS')
        if self.APRENDE_PERDER.search(normalized): neg_score += 4.5; reasons.append('APRENDE_PERDER')
        if self.MOCHILA_PASADO.search(normalized): neg_score += 5.5; reasons.append('MOCHILA_OSCURA')
        if self.JAPONESA_DESPECTIVO.search(normalized): neg_score += 5.0; reasons.append('JAPONESA_DESP')
        if self.SACAR_VUELTA_LEY.search(normalized): neg_score += 5.0; reasons.append('EVADE_LEY')
        if self.ASOCIACION_PERDER.search(normalized): neg_score += 4.0; reasons.append('ASOC_PERDER')
        if self.LUCHADOR_SOCIAL.search(normalized): neg_score += 4.5; reasons.append('LUCHADOR_SOCIAL')
        if self.APOYO_MENDOZA.search(normalized): neg_score += 5.0; reasons.append('APOYO_MENDOZA')

        # --- pos_score: 26 patrones (CORRECCIÓN 10: 12 patrones retirados) ---
        if self.INSULTOS_CASTILLO.search(normalized): pos_score += 6.5; reasons.append('INSULTOS_CASTILLO')
        if self.CRITICA_CASTILLO.search(normalized): pos_score += 6.0; reasons.append('CRITICA_CASTILLO')
        if self.TERRORISMO.search(normalized): pos_score += 6.0; reasons.append('TERRORISMO')
        if self.ORDEN_CAOS.search(normalized): pos_score += 6.0; reasons.append('ORDEN_CAOS')
        if self.ANTI_COMUNISMO.search(normalized): pos_score += 5.5; reasons.append('ANTI_COMUNISMO')
        if self.VENEZUELA_CUBA.search(normalized): pos_score += 5.5; reasons.append('VENEZUELA_CUBA')
        if self.APOYO_KEIKO.search(normalized): pos_score += 5.0; reasons.append('APOYO_KEIKO')
        if self.PREGUNTA_RETORICA_CRITICA.search(normalized): pos_score += 4.5; reasons.append('PREGUNTA_RET')
        if self.ERROR_MATEMATICO.search(normalized): pos_score += 4.5; reasons.append('ERROR_MAT')
        if self.PAIS_DESARROLLADO_SARCASMO.search(normalized): pos_score += 4.0; reasons.append('DUBAI_SARCASMO')
        if self.MIEDO_ECONOMICO.search(normalized): pos_score += 4.0; reasons.append('MIEDO_ECON')
        if self.ESTABILIDAD_ECONOMICA.search(normalized): pos_score += 4.0; reasons.append('ESTABILIDAD')
        if self.ERROR_FACTUAL.search(normalized): pos_score += 4.0; reasons.append('ERROR_FACTUAL')
        if self.AUTORITARIO_DESPIADADO.search(normalized): pos_score += 4.0; reasons.append('AUTORITARIO')
        if self.SARCASMO_IRONIA.search(normalized): pos_score += 3.5; reasons.append('SARCASMO')
        if self.EVASION_RESPUESTA.search(normalized): pos_score += 3.5; reasons.append('EVASION_RESP')
        if self.VOTO_CASTIGO.search(normalized): pos_score += 3.5; reasons.append('VOTO_CASTIGO')
        if self.BUSCANDO_PUESTO.search(normalized): pos_score += 3.5; reasons.append('BUSCA_CHAMBA')
        if self.OPORTUNISTA.search(normalized): pos_score += 3.5; reasons.append('OPORTUNISTA')
        if self.PREGUNTA_RETORICA_PROPUESTAS.search(normalized): pos_score += 5.5; reasons.append('PREGUNTA_PROPUESTAS')
        if self.ERROR_PBI_PORCENTAJE.search(normalized): pos_score += 6.0; reasons.append('ERROR_PBI')
        if self.VICTORIA_KEIKO_DEBATE.search(normalized): pos_score += 6.5; reasons.append('KEIKO_BARRIO')
        if self.OPORTUNISTA_CASTILLO.search(normalized): pos_score += 4.5; reasons.append('OPORTUNISTA')
        if self.FELPUDO_INSULTOS.search(normalized) and re.search(r'\bsheput\b', normalized, re.I):
            pos_score += 3.5; reasons.append('INSULTO_SHEPUT')
        if self.CRITICA_MENDOZA.search(normalized): pos_score += 5.0; reasons.append('CRITICA_MENDOZA')
        if self.APOYO_RLA_DESOTO.search(normalized): pos_score += 5.0; reasons.append('APOYO_MERCADO_1V')
        if self.CRITICA_LESCANO.search(normalized): pos_score += 3.0; reasons.append('CRITICA_LESCANO')

        neg_score, pos_score = self._sumar_con_direccion(
            normalized, self.EMOCION_NEGATIVA, 4.5, 'EMOCION_NEG', neg_score, pos_score, reasons)
        neg_score, pos_score = self._sumar_con_direccion(
            normalized, self.INSULTOS_GENERALES, 4.0, 'INSULTOS', neg_score, pos_score, reasons)
        neg_score, pos_score = self._sumar_con_direccion(
            normalized, self.SLANG_NEGATIVO, 4.0, 'SLANG_NEG', neg_score, pos_score, reasons)
        neg_score, pos_score = self._sumar_con_direccion(
            normalized, self.CRITICA_INSTITUCIONAL_TERM, 4.0, 'CRITICA_INST', neg_score, pos_score, reasons)
        neg_score, pos_score = self._sumar_con_direccion(
            normalized, self.BURLA_RISA_TERM, 1.5, 'BURLA_RISA', neg_score, pos_score, reasons)

        if self.ANALISIS_TECNICO.search(normalized): neutral_score += 3.5; reasons.append('ANALISIS_TEC')
        if self.PREGUNTA_GENUINA.search(normalized): neutral_score += 2.5; reasons.append('PREGUNTA_GEN')

        total = neg_score + pos_score + neutral_score
        political_strength = self._get_political_content_strength(normalized)

        if total == 0:
            return {'sentiment': 'NEUTRAL', 'confidence': 0.0, 'score_neg': 0,
                    'score_pos': 0, 'score_neutral': 0, 'reason': 'SIN_PATRONES',
                    'political_strength': political_strength}

        neg_ratio, pos_ratio, neutral_ratio = neg_score / total, pos_score / total, neutral_score / total
        max_score = max(neg_score, pos_score, neutral_score)
        second_max = sorted([neg_score, pos_score, neutral_score])[-2]
        diff_ratio = (max_score - second_max) / total if total > 0 else 0

        if diff_ratio < 0.15:
            return {'sentiment': 'NEUTRAL', 'confidence': 0.6, 'score_neg': neg_score,
                    'score_pos': pos_score, 'score_neutral': neutral_score,
                    'reason': 'AMBIGUO', 'political_strength': political_strength}

        if neg_score > pos_score and neg_score > neutral_score:
            return {'sentiment': 'NEGATIVE', 'confidence': min(0.95, neg_ratio + 0.05),
                    'score_neg': neg_score, 'score_pos': pos_score, 'score_neutral': neutral_score,
                    'reason': ' | '.join(reasons[:3]), 'political_strength': political_strength}
        elif pos_score > neg_score and pos_score > neutral_score:
            return {'sentiment': 'POSITIVE', 'confidence': min(0.95, pos_ratio + 0.05),
                    'score_neg': neg_score, 'score_pos': pos_score, 'score_neutral': neutral_score,
                    'reason': ' | '.join(reasons[:3]), 'political_strength': political_strength}
        else:
            self.metrics['neutral_real'] += 1
            return {'sentiment': 'NEUTRAL', 'confidence': min(0.85, neutral_ratio + 0.15),
                    'score_neg': neg_score, 'score_pos': pos_score, 'score_neutral': neutral_score,
                    'reason': ' | '.join(reasons[:3]), 'political_strength': political_strength}

    def analyze_model_calibrated(self, text):
        if not self.model_loaded:
            return {'sentiment': 'NEUTRAL', 'confidence': 0.0, 'margen': 1.0}
        try:
            resultados = self.sentiment_pipeline(text[:512])[0]
            resultados_ordenados = sorted(resultados, key=lambda x: x['score'], reverse=True)
            label = resultados_ordenados[0]['label'].upper()
            conf = resultados_ordenados[0]['score']
            margen = resultados_ordenados[0]['score'] - resultados_ordenados[1]['score']

            political_strength = self._get_political_content_strength(text)
            if political_strength > 0.3:
                conf = max(conf * (1.0 - political_strength * 0.6), 0.3)

            if label in ['POSITIVE', 'POSITIVO', 'LABEL_2']:
                sentimiento = 'POSITIVE'
            elif label in ['NEGATIVE', 'NEGATIVO', 'LABEL_0']:
                sentimiento = 'NEGATIVE'
            else:
                sentimiento = 'NEUTRAL'
            return {'sentiment': sentimiento, 'confidence': conf, 'margen': margen}
        except Exception:
            return {'sentiment': 'NEUTRAL', 'confidence': 0.0, 'margen': 1.0}

    def analyze_hybrid(self, text):
        cache_key = text.strip().lower() if isinstance(text, str) else str(text)
        if cache_key in self.cache:
            return self.cache[cache_key]

        self.metrics['total'] += 1

        if not isinstance(text, str) or self._contenido_insuficiente(text):
            result = {'sentimiento_economico': 'NEUTRAL', 'sentimiento_numerico': 0,
                      'confianza_sentimiento': 0.0, 'metodo_clasificacion': 'CONTENIDO_INSUFICIENTE',
                      'score_anti_keiko': 0, 'score_anti_castillo': 0, 'score_neutral': 0,
                      'confianza_reglas': 0.0, 'confianza_modelo': 0.0,
                      'fuerza_contenido_politico': 0.0, 'margen_modelo': None,
                      'razones': 'CONTENIDO_INSUFICIENTE'}
            self.metrics['neutral_real'] += 1
            self.cache[cache_key] = result
            return result

        rules_result = self.analyze_rules(text)
        political_strength = rules_result.get('political_strength', 0.0)

        if not self.model_loaded:
            self.metrics['rules_only'] += 1
            result = self._format_result(rules_result, 'SOLO_REGLAS')
            self.cache[cache_key] = result
            return result

        model_result = self.analyze_model_calibrated(text)
        texto_normalizado = self._normalize_text(text)
        if model_result['sentiment'] in ('POSITIVE', 'NEGATIVE'):
            model_result['sentiment'] = self._reorientar_polaridad_modelo(texto_normalizado, model_result['sentiment'])

        rules_conf = rules_result['confidence']
        model_conf = model_result['confidence']
        model_margen = model_result.get('margen', 1.0)

        if political_strength > 0.5:
            final_sentiment = rules_result['sentiment']
            final_confidence = min(rules_conf + 0.15, 0.95)
            method = 'VETO_POLITICO_FUERTE'
            self.metrics['political_override'] += 1
            self.metrics['rules_only'] += 1
        elif political_strength > 0.3:
            if rules_conf >= 0.6:
                final_sentiment = rules_result['sentiment']
                final_confidence = rules_conf
                method = 'REGLAS_POLITICAS_MODERADAS'
                self.metrics['rules_only'] += 1
            else:
                if rules_result['sentiment'] == model_result['sentiment']:
                    final_sentiment = rules_result['sentiment']
                    final_confidence = (rules_conf + model_conf) / 2
                    method = 'HIBRIDO_COINCIDEN_POLITICO'
                    self.metrics['hybrid'] += 1
                else:
                    final_sentiment = rules_result['sentiment']
                    final_confidence = rules_conf * 1.1
                    method = 'REGLAS_PRIORIDAD_POLITICA'
                    self.metrics['rules_only'] += 1
        else:
            if rules_conf >= 0.7:
                final_sentiment = rules_result['sentiment']
                final_confidence = rules_conf
                method = 'REGLAS_FUERTES'
                if model_result['sentiment'] == rules_result['sentiment']:
                    final_confidence = min(final_confidence + 0.12, 0.98)
                    method = 'HIBRIDO_COINCIDEN'
                    self.metrics['hybrid'] += 1
                else:
                    self.metrics['rules_only'] += 1
            elif rules_result['sentiment'] == 'NEUTRAL':
                bando_detectado = self._detectar_bando_mencionado(texto_normalizado)
                if (political_strength < 0.15 and model_conf >= 0.78
                        and bando_detectado is not None and model_margen >= self.MARGEN_MINIMO):
                    final_sentiment = model_result['sentiment']
                    final_confidence = model_conf
                    method = 'MODELO_FUERTE'
                    self.metrics['model_used'] += 1
                else:
                    final_sentiment = 'NEUTRAL'
                    final_confidence = max(0.4, rules_conf)
                    if (bando_detectado is not None and model_margen < self.MARGEN_MINIMO
                            and political_strength < 0.15 and model_conf >= 0.78):
                        method = 'NEUTRAL_MARGEN_BAJO'
                        self.metrics['neutral_margen_bajo'] += 1
                    else:
                        method = 'NEUTRAL_CONSERVADOR'
                    self.metrics['rules_only'] += 1
            elif rules_conf > model_conf:
                final_sentiment = rules_result['sentiment']
                final_confidence = rules_conf
                method = 'REGLAS_PRIORIDAD'
                self.metrics['rules_only'] += 1
            else:
                bando_detectado_2 = self._detectar_bando_mencionado(texto_normalizado)
                margen_suficiente = model_result.get('margen', 1.0) >= self.MARGEN_MINIMO
                if bando_detectado_2 is not None and margen_suficiente:
                    final_sentiment = model_result['sentiment']
                    final_confidence = model_conf
                    method = 'MODELO_PRIORIDAD'
                    self.metrics['model_used'] += 1
                elif bando_detectado_2 is not None and not margen_suficiente:
                    final_sentiment = 'NEUTRAL'
                    final_confidence = max(0.4, rules_conf)
                    method = 'NEUTRAL_MARGEN_BAJO'
                    self.metrics['rules_only'] += 1
                else:
                    final_sentiment = 'NEUTRAL'
                    final_confidence = max(0.4, rules_conf)
                    method = 'NEUTRAL_SIN_CANDIDATO'
                    self.metrics['rules_only'] += 1

        final_confidence = max(0.1, min(0.99, final_confidence))
        sentiment_map = {'POSITIVE': 'PRO_KEIKO_ANTI_CASTILLO', 'NEGATIVE': 'PRO_CASTILLO_ANTI_KEIKO', 'NEUTRAL': 'NEUTRAL'}

        result = {
            'sentimiento_economico': sentiment_map.get(final_sentiment, final_sentiment),
            'sentimiento_numerico': self._get_numeric_sentiment(final_sentiment),
            'confianza_sentimiento': round(final_confidence, 3),
            'metodo_clasificacion': method,
            'score_anti_keiko': rules_result.get('score_neg', 0),
            'score_anti_castillo': rules_result.get('score_pos', 0),
            'score_neutral': rules_result.get('score_neutral', 0),
            'confianza_reglas': round(rules_conf, 3),
            'confianza_modelo': round(model_conf, 3),
            'fuerza_contenido_politico': round(political_strength, 3),
            'margen_modelo': round(model_margen, 3),
            'razones': rules_result.get('reason', 'N/A')
        }
        self.cache[cache_key] = result
        return result

    def _get_numeric_sentiment(self, sentiment):
        return {'POSITIVE': 1, 'NEGATIVE': -1, 'NEUTRAL': 0}.get(sentiment, 0)

    def _format_result(self, rules_result, method):
        sentiment_map = {'POSITIVE': 'PRO_KEIKO_ANTI_CASTILLO', 'NEGATIVE': 'PRO_CASTILLO_ANTI_KEIKO', 'NEUTRAL': 'NEUTRAL'}
        sentiment = rules_result['sentiment']
        return {
            'sentimiento_economico': sentiment_map.get(sentiment, sentiment),
            'sentimiento_numerico': self._get_numeric_sentiment(sentiment),
            'confianza_sentimiento': round(rules_result['confidence'], 3),
            'metodo_clasificacion': method,
            'score_anti_keiko': rules_result.get('score_neg', 0),
            'score_anti_castillo': rules_result.get('score_pos', 0),
            'score_neutral': rules_result.get('score_neutral', 0),
            'confianza_reglas': round(rules_result['confidence'], 3),
            'confianza_modelo': 0.0,
            'fuerza_contenido_politico': round(rules_result.get('political_strength', 0), 3),
            'margen_modelo': None,
            'razones': rules_result.get('reason', 'N/A')
        }


def procesar_dataset(archivo_path, columna_texto='tweet_homologado', usar_modelo=True):
    try:
        df = pd.read_excel(archivo_path)
        print(f"✓ Archivo cargado: {len(df):,} tweets")
    except Exception as e:
        print(f"✗ Error: {e}")
        return None

    if columna_texto not in df.columns:
        print(f"✗ Columna '{columna_texto}' no existe")
        return None

    columnas_resultado = [
        'sentimiento_economico', 'sentimiento_numerico', 'confianza_sentimiento',
        'metodo_clasificacion', 'score_anti_keiko', 'score_anti_castillo',
        'score_neutral', 'confianza_reglas', 'confianza_modelo',
        'fuerza_contenido_politico', 'margen_modelo', 'razones'
    ]
    renombrar = {c: f'{c}_prev' for c in columnas_resultado if c in df.columns}
    if renombrar:
        df = df.rename(columns=renombrar)

    analyzer = PeruvianSentimentClassifier(use_model=usar_modelo)

    print("\nProcesando tweets...")
    results = []
    total = len(df)
    for idx, row in df.iterrows():
        results.append(analyzer.analyze_hybrid(row[columna_texto]))
        if (idx + 1) % 5000 == 0:
            print(f"  {idx + 1:,}/{total:,} ({((idx+1)/total)*100:.1f}%)")

    print(f"✓ {total:,} tweets procesados\n")
    results_df = pd.DataFrame(results)
    return pd.concat([df.reset_index(drop=True), results_df], axis=1)


if __name__ == "__main__":
    try:
        from google.colab import files
        print("Sube tu archivo Excel...")
        uploaded = files.upload()
        if uploaded:
            archivo_nombre = list(uploaded.keys())[0]
            with open(archivo_nombre, 'wb') as f:
                f.write(uploaded[archivo_nombre])
            df_resultado = procesar_dataset(archivo_nombre, columna_texto='tweet_homologado', usar_modelo=True)
            if df_resultado is not None:
                archivo_salida = archivo_nombre.replace('.xlsx', '_V3_BALANCEADO.xlsx')
                df_resultado.to_excel(archivo_salida, index=False)
                files.download(archivo_salida)
                print("✓ Completado")
    except ImportError:
        print("Ejecutar en Google Colab o adaptar para local")

Sube tu archivo Excel...


Saving tweets_limpios (3).xlsx to tweets_limpios (3).xlsx
✓ Archivo cargado: 97,118 tweets
Cargando modelo ML...


config.json:   0%|          | 0.00/975 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.11GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

...

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


  5,000/97,118 (5.1%)
  10,000/97,118 (10.3%)
  15,000/97,118 (15.4%)
  20,000/97,118 (20.6%)
  25,000/97,118 (25.7%)
  30,000/97,118 (30.9%)
  35,000/97,118 (36.0%)
  40,000/97,118 (41.2%)
  45,000/97,118 (46.3%)
  50,000/97,118 (51.5%)
  55,000/97,118 (56.6%)
  60,000/97,118 (61.8%)
  65,000/97,118 (66.9%)
  70,000/97,118 (72.1%)
  75,000/97,118 (77.2%)
  80,000/97,118 (82.4%)
  85,000/97,118 (87.5%)
  90,000/97,118 (92.7%)
  95,000/97,118 (97.8%)
✓ 97,118 tweets procesados



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ Completado
